## CIFAR10 Hessian Plots

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
"""python scripts/cifar10_hessian.py \
    --ckpt_dir "/content/drive/MyDrive/moe_project/checkpoints/SoftMoE/E50-X4" \
    --data_dir "./data" \
    --num_train 2000 \
    --num_test 2000
    """

In [ ]:
base = "../checkpoints"
models = {
    "Dense":     os.path.join(base, "Dense",     "E50"),
    "SoftMoE":   os.path.join(base, "SoftMoE",   "E50-X4"),
    "SparseMoE": os.path.join(base, "SparseMoE", "E50-X4-K2"),
}

def load_hessian(ckpt_dir):
    return torch.load(os.path.join(ckpt_dir, "hessian.pt"), map_location="cpu")

hessian_data = {name: load_hessian(path) for name, path in models.items()}


In [ ]:
labels = list(hessian_data.keys())
n_models = len(labels)

# Collect stats
lambda_train = [hessian_data[m]["hessian"]["train"]["lambda_max"] for m in labels]
lambda_test  = [hessian_data[m]["hessian"]["test"]["lambda_max"]  for m in labels]
trace_train  = [hessian_data[m]["hessian"]["train"]["trace"]      for m in labels]
trace_test   = [hessian_data[m]["hessian"]["test"]["trace"]       for m in labels]

# 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
ax1, ax2, ax3, ax4 = axes.flat

x = np.arange(n_models)
width = 0.35

# -------- (1) Top eigenvalue: train vs test --------
ax1.bar(x - width/2, lambda_train, width, label="train")
ax1.bar(x + width/2, lambda_test,  width, label="test")
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Top eigenvalue λ₁")
ax1.set_title("Top Hessian eigenvalue")
ax1.legend()

# -------- (2) Trace: train vs test --------
ax2.bar(x - width/2, trace_train, width, label="train")
ax2.bar(x + width/2, trace_test,  width, label="test")
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylabel("Tr(H)")
ax2.set_title("Hessian trace")
ax2.legend()

# -------- (3) ESD (train) --------
for m in labels:
    es = hessian_data[m]["hessian"]["train"]["density_eigs"]
    ws = hessian_data[m]["hessian"]["train"]["density_weights"]
    ax3.plot(es, ws, label=m)

ax3.set_xlabel("Eigenvalue")
ax3.set_ylabel("Density")
ax3.set_title("ESD (train)")
ax3.legend()

# -------- (4) Curvature (loss vs λ along top eigenvector, train) --------
for m in labels:
    alphas = np.array(hessian_data[m]["hessian"]["train"]["curvature_alphas"])
    losses = np.array(hessian_data[m]["hessian"]["train"]["curvature_losses"])
    ax4.plot(alphas, losses, marker="o", label=m)

ax4.axvline(0.0, linestyle="--")
ax4.set_xlabel("λ")
ax4.set_ylabel("Loss")
ax4.set_title("Loss along top eigenvector (train)")
ax4.legend()

plt.tight_layout()
plt.show()


In [ ]:
#### SAME AS ABOVE ALTERNATE STYLE
# --------- Style settings ---------
plt.style.use("seaborn-v0_8-whitegrid")   # clean academic look

TITLE_FONTSIZE = 15
LABEL_FONTSIZE = 13
TICK_FONTSIZE  = 12
LEGEND_FONTSIZE = 11

# consistent color palette for Dense / SoftMoE / SparseMoE
colors = {
    "Dense":     "#1f77b4",   # blue
    "SoftMoE":   "#ff7f0e",   # orange
    "SparseMoE": "#2ca02c",   # green
}

labels = list(hessian_data.keys())
n_models = len(labels)

# Collect stats
lambda_train = [hessian_data[m]["hessian"]["train"]["lambda_max"] for m in labels]
lambda_test  = [hessian_data[m]["hessian"]["test"]["lambda_max"]  for m in labels]
trace_train  = [hessian_data[m]["hessian"]["train"]["trace"]      for m in labels]
trace_test   = [hessian_data[m]["hessian"]["test"]["trace"]       for m in labels]

# 2x2 layout
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
ax1, ax2, ax3, ax4 = axes.flat

x = np.arange(n_models)
width = 0.35

# ----------------------------------------------------
# (1) Top eigenvalue: train vs test
# ----------------------------------------------------
for i, m in enumerate(labels):
    ax1.bar(x[i] - width/2, lambda_train[i], width, 
            label="train" if i == 0 else "", color=colors[m], alpha=0.85)
    ax1.bar(x[i] + width/2, lambda_test[i],  width, 
            label="test"  if i == 0 else "", 
            color=colors[m], alpha=0.35)   # lighter shade for test

ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=TICK_FONTSIZE)
ax1.set_ylabel("Top eigenvalue λ₁", fontsize=LABEL_FONTSIZE)
ax1.set_title("Top Hessian Eigenvalue (Train vs Test)", fontsize=TITLE_FONTSIZE)
ax1.legend(fontsize=LEGEND_FONTSIZE)

# ----------------------------------------------------
# (2) Trace(H): train vs test
# ----------------------------------------------------
for i, m in enumerate(labels):
    ax2.bar(x[i] - width/2, trace_train[i], width, 
            label="train" if i == 0 else "", color=colors[m], alpha=0.85)
    ax2.bar(x[i] + width/2, trace_test[i],  width,
            label="test"  if i == 0 else "", 
            color=colors[m], alpha=0.35)

ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=TICK_FONTSIZE)
ax2.set_ylabel("Trace(H)", fontsize=LABEL_FONTSIZE)
ax2.set_title("Hessian Trace (Train vs Test)", fontsize=TITLE_FONTSIZE)
ax2.legend(fontsize=LEGEND_FONTSIZE)

# ----------------------------------------------------
# (3) ESD (train)
# ----------------------------------------------------
for m in labels:
    es = hessian_data[m]["hessian"]["train"]["density_eigs"]
    ws = hessian_data[m]["hessian"]["train"]["density_weights"]
    ax3.plot(es, ws, label=m, color=colors[m], linewidth=2)

ax3.set_xlabel("Eigenvalue", fontsize=LABEL_FONTSIZE)
ax3.set_ylabel("Density", fontsize=LABEL_FONTSIZE)
ax3.set_title("Empirical Spectral Density (Train)", fontsize=TITLE_FONTSIZE)
ax3.legend(fontsize=LEGEND_FONTSIZE)
ax3.tick_params(axis='both', labelsize=TICK_FONTSIZE)

# ----------------------------------------------------
# (4) Curvature: loss vs λ
# ----------------------------------------------------
for m in labels:
    alphas = np.array(hessian_data[m]["hessian"]["train"]["curvature_alphas"])
    losses = np.array(hessian_data[m]["hessian"]["train"]["curvature_losses"])
    ax4.plot(alphas, losses, marker="o", label=m, color=colors[m], linewidth=2, markersize=5)

ax4.axvline(0.0, linestyle="--", color="gray", linewidth=1)
ax4.set_xlabel("λ (step along top eigenvector)", fontsize=LABEL_FONTSIZE)
ax4.set_ylabel("Loss", fontsize=LABEL_FONTSIZE)
ax4.set_title("Loss Landscape Curvature (Train)", fontsize=TITLE_FONTSIZE)
ax4.legend(fontsize=LEGEND_FONTSIZE)
ax4.tick_params(axis='both', labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.show()
